In [1]:
import pandas as pd 


# 读取 Feather 文件
data = pd.read_feather(rf'/project/model_share/share_1_new/factor_csy/fac20250721/merged_data.fea')

print("✅ Feather 文件读取完成！")


✅ Feather 文件读取完成！


In [2]:
data.head(10)

,date,Code,a0001_factor_fea_20250630,a0002_factor_fea_20250630,a0003_factor_fea_20250630,a0004_factor_fea_20250630,a0005_factor_fea_20250630,a0006_factor_fea_20250630,a0007_factor_fea_20250630,a0008_factor_fea_20250630,...,fac3035,fac3036,fac3037,fac3038,fac3039,fac3040,fac3041,fac3042,fac3043,fac3044
0,20210105,000001,0.438252,0.238076,-0.526136,0.809813,-0.496128,1.251059,-0.517327,-0.322449,...,-0.110591,0.050464,0.064927,0.081030,0.095168,0.101904,0.127329,0.043079,0.027992,-0.084919
1,20210105,000002,0.560824,0.345450,-0.000796,-0.190782,0.148574,1.324764,0.147143,-0.169908,...,-0.228317,-0.036660,-0.110984,-0.032141,-0.101185,-0.002097,-0.001209,0.002828,-0.094783,-0.222358
2,20210105,000004,0.124771,-0.716989,-0.245919,-0.690275,0.473380,-0.145338,1.134138,0.684091,...,0.055253,-0.046485,-0.041135,-0.077998,-0.073629,-0.206159,-0.227427,-0.225368,-0.180979,0.049022
3,20210105,000005,0.000000,-1.345319,-0.979040,-2.109435,-0.685095,0.000000,-1.176065,-1.347414,...,-0.109035,-0.070376,-0.042297,-0.065443,-0.039849,-0.052984,-0.050973,-0.050437,-0.062537,-0.094527
4,20210105,000006,-1.926162,-0.902478,-1.627878,-1.450586,-0.793003,-1.223660,-0.107143,-0.298837,...,-6.354914,-2.125352,-2.111854,-1.813738,-1.803210,0.516527,0.632701,1.607664,0.117812,-6.511986
5,20210105,000007,0.000000,0.000000,0.000000,0.000000,-0.958644,0.000000,0.000000,-0.592691,...,-0.689772,-0.476745,-0.534680,-0.442091,-0.496562,-0.095907,-0.093943,0.268501,-0.065188,-0.759542
6,20210105,000008,-1.674689,-1.610272,-2.190131,-0.919477,0.172273,-2.006026,1.535360,0.770608,...,-0.116423,-0.030849,-0.034073,-0.034950,-0.038188,-0.062829,-0.065853,-0.070390,-0.075561,-0.098238
7,20210105,000009,-0.291506,-0.845572,-1.447605,-0.690400,-0.175880,-1.104696,-0.284196,-0.292768,...,0.734418,-0.011135,-0.014088,-0.008762,-0.011626,-0.418605,-0.391069,-0.415001,-0.243183,0.628947
8,20210105,000010,-0.968894,-0.503371,-1.994338,-1.674059,1.643491,-1.685057,-0.175107,1.853964,...,0.028831,0.077215,0.099959,0.065808,0.086970,0.058103,0.047798,-0.015257,-0.002337,0.015200
9,20210105,000011,-0.493728,-0.608667,-0.737641,-0.979190,1.315775,0.141545,-0.366610,1.252828,...,-0.194839,-0.148061,-0.014604,-0.134712,-0.011944,-0.042732,-0.039240,-0.063735,-0.069914,-0.188551


In [12]:

import pandas as pd
import concurrent.futures

# 假设 data 已经是你的 DataFrame
fac_cols = [col for col in data.columns if col.startswith("fac") or col.startswith("a")]

# 并行任务函数
def classify_column(col):
    sample = data[col].dropna().sample(n=1000000, random_state=42) if data[col].count() > 10000000 else data[col].dropna()
    unique_vals = sample.nunique()
    dtype = data[col].dtype

    if dtype == 'object' or unique_vals <= 1000:
        return (col, 'categorical')
    else:
        return (col, 'continuous')



In [13]:
len(fac_cols)

3298

In [14]:
# 并发处理
categorical = []
continuous = []

with concurrent.futures.ThreadPoolExecutor(max_workers=128) as executor:
    results = list(executor.map(classify_column, fac_cols))

# 分类汇总
for col, col_type in results:
    if col_type == 'categorical':
        categorical.append(col)
    else:
        continuous.append(col)

# 输出结果
print(f"类别特征数: {len(categorical)}")
print(f"连续特征数: {len(continuous)}")

类别特征数: 148
连续特征数: 3150


In [15]:
# 保存为 txt
with open('categorical_features.txt', 'w') as f:
    for col in categorical:
        f.write(col + '\n')

with open('continuous_features.txt', 'w') as f:
    for col in continuous:
        f.write(col + '\n')


In [ ]:
with open('categorical_features.txt', 'r') as f:
    categorical = [line.strip() for line in f.readlines()]

with open('continuous_features.txt', 'r') as f:
    continuous = [line.strip() for line in f.readlines()]

In [16]:
# 识别并保留非 fac 列
id_cols = ['Code', 'date']  # 如果你的列名是这样，确保实际列名拼写一致

# 添加 identifier 列（如果存在）
data_float = data[id_cols + continuous]
data_int = data[id_cols + categorical]

In [17]:
data_float.to_feather('/project/model_share/share_1_new/factor_csy/fac20250721/data_float.fea')
data_int.to_feather('/project/model_share/share_1_new/factor_csy/fac20250721/data_int.fea')

In [ ]:
import pandas as pd
data_float = pd.read_feather('/project/model_share/share_1_new/factor_csy/int_or_float/data_float.fea')
data_int = pd.read_feather('/project/model_share/share_1_new/factor_csy/int_or_float/data_int.fea')

In [11]:
import pandas as pd

# 取出第一行（可以用 .head(1)）
float_first_row = data_float.head(100)
int_first_row = data_int.head(100)

# 保存为 Excel
with pd.ExcelWriter("feature_check.xlsx") as writer:
    float_first_row.to_excel(writer, sheet_name="data_float", index=False)
    int_first_row.to_excel(writer, sheet_name="data_int", index=False)


/tmp/ipykernel_2011850/2684768006.py:8: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  with pd.ExcelWriter("feature_check.xlsx") as writer:


In [21]:
data_float[:10]


,Code,date,fac0001,fac0002,fac0003,fac0004,fac0005,fac0006,fac0007,fac0008,...,fac3035,fac3036,fac3037,fac3038,fac3039,fac3040,fac3041,fac3042,fac3043,fac3044
0,000001,20210104,0.103297,0.508136,-0.276862,-4.331658,0.261553,-5.413435,0.634146,-4.365740,...,0.009834,0.016627,0.014436,0.016538,0.014353,0.020904,0.020863,0.039058,0.032413,0.007140
1,000002,20210104,0.041108,0.692095,-0.148505,-3.625367,0.389545,3.513462,1.249417,-3.622822,...,0.004653,0.011976,0.012147,0.010844,0.011016,0.017384,0.016231,0.023557,0.018286,0.004495
2,000004,20210104,0.061697,0.900875,-0.046951,-1.091614,0.458014,0.000000,1.000000,-1.039020,...,-0.010993,-0.015484,-0.012007,-0.024155,-0.020684,-0.026982,-0.035714,-0.039314,-0.024251,-0.008712
3,000005,20210104,0.000000,0.932106,-0.026149,0.495133,0.558257,-2.639419,0.459459,0.573825,...,0.011329,-0.000568,0.002651,-0.000432,0.002787,-0.018660,-0.018512,-0.034847,-0.012931,0.009670
4,000006,20210104,0.032552,0.958341,-0.025876,-1.673883,0.331534,-3.014057,0.388889,-1.567781,...,-0.015754,0.013279,0.034034,0.013207,0.033960,0.036399,0.036317,0.056604,0.054037,-0.014980
5,000007,20210104,0.541024,0.000000,-0.994693,-1.826237,0.000000,-0.836105,0.813559,-1.848190,...,-0.147636,-0.154539,-0.153686,-0.171622,-0.170444,0.004918,-0.013638,0.208612,0.033461,-0.142704
6,000008,20210104,0.038402,0.811723,-0.095611,-2.963371,0.405330,-3.766941,0.432836,-2.908616,...,0.001008,-0.006728,-0.007232,-0.008223,-0.008727,-0.023417,-0.024910,-0.033701,-0.017139,0.000868
7,000009,20210104,0.182027,0.719200,-0.191238,-1.913540,0.228649,-3.785592,0.574803,-1.817809,...,-0.008438,-0.047177,-0.041414,-0.045641,-0.039880,-0.054035,-0.052467,-0.067854,-0.055058,-0.007033
8,000010,20210104,0.144204,0.639520,0.148647,1.195885,1.939908,-3.978168,0.546763,1.199352,...,-0.022014,-0.025434,-0.045672,-0.036448,-0.056679,-0.044965,-0.056138,-0.059566,-0.060314,-0.012184
9,000011,20210104,0.043398,0.919260,-0.058934,-3.149672,0.228621,-3.194936,0.450980,-3.128270,...,-0.016761,-0.010241,0.002024,-0.010429,0.001836,0.005072,0.004882,0.004071,0.001035,-0.014538
